# Spotlight Your Instructions: Instruction-following with Dynamic Attention Steering

**Paper**: [Venakteswaran and Contractor, EACL 2026](https://aclanthology.org/2026.eacl-long.174/)

## What is Spotlight?

Spotlight is an **inference-time attention steering mechanism** that biases how a language model attends to emphasized text spans in prompts. By directing attention toward key instruction phrases, we can increase the model's likelihood of following those instructions without fine-tuning.

## What This Notebook Shows

1. Load Qwen2-1.5B-Instruct with the Spotlight worker
2. Generate baseline output (no steering)
3. Generate with Spotlight active (attention steering)
4. Compare the outputs

## Performance Note

⚠️ **Spotlight requires `enforce_eager=True`, which disables Flash Attention.**

This is a necessary trade-off because Flash Attention fuses Q, K, V into a single kernel, preventing hook access to intermediate tensors needed for attention steering.

## Installation & Setup

In [1]:
print("\nInstalling vLLM (with CUDA support)...")
%pip install --upgrade vllm

print("\nInstalling vllm_hook_plugins...")
%pip install -e ../vllm_hook_plugins

print("\nInstalling requirements...")
%pip install -r ../requirement.txt

print("\n✓ Installation complete")



Installing vLLM (with CUDA support)...
Note: you may need to restart the kernel to use updated packages.

Installing vllm_hook_plugins...
Obtaining file:///home/danish/projects/vLLM-Hook/vllm_hook_plugins
  Preparing metadata (setup.py) ... done
  Attempting uninstall: vllm-hook-plugins
    Found existing installation: vllm-hook-plugins 0.1.0
    Uninstalling vllm-hook-plugins-0.1.0:
      Successfully uninstalled vllm-hook-plugins-0.1.0
  DEPRECATION: Legacy editable install of vllm-hook-plugins==0.1.0 from file:///home/danish/projects/vLLM-Hook/vllm_hook_plugins (setup.py develop) is deprecated. pip 25.3 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  Runn

## Imports & Setup

In [1]:
import os
import multiprocessing as mp
import torch
from vllm import SamplingParams

# Setup vLLM V1 engine
mp.set_start_method("spawn", force=True)
os.environ["VLLM_USE_V1"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Import Spotlight
from vllm_hook_plugins import HookLLM, generate_with_spotlight, register_plugins
register_plugins()

print("Environment configured")

/home/danish/miniforge3/envs/mainlinux/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WARNING 04-16 17:51:41 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
Environment configured


## Load Model

In [ ]:
MODEL = "Qwen/Qwen2-1.5B-Instruct"
cache_dir = os.path.expanduser("~/.cache")

print(f"Loading {MODEL}...")
print("This may take 1-2 minutes on first run.\n")

llm = HookLLM(
    model=MODEL,
    worker_name="probe_spotlight",
    download_dir=cache_dir,
    gpu_memory_utilization=0.8,
    dtype=torch.float16,
    enable_hook=True,
    enforce_eager=True,  # Required for Spotlight (disables Flash Attention)
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
)

print(f"Model loaded: {MODEL}")
print(f"Tokenizer ready")
print(f"Spotlight worker enabled")

## Configure Test Parameters

In [3]:
# ========================================================
# Modify these parameters to test different prompts
# ========================================================

prompt = "Return the response for the following as a JSON: Write a 400 word paragraph on France and its food specifically focussing on dishes. Include the paragraph and the list of dishes mentioned the paragraph in seperate fields."
emph_strings = ["Return the response for the following as a JSON:"]  # Emphasize this part of the prompt
alpha = 0.1  # Target 10% attention on emphasized span
max_tokens = 500
temperature = 0.0

# ========================================================

print(f"Prompt:      {prompt}")
print(f"Emphasis:    {emph_strings}")
print(f"Alpha:       {alpha}")
print(f"Max tokens:  {max_tokens}")
print(f"Temperature: {temperature}")

Prompt:      Return the response for the following as a JSON: Write a 400 word paragraph on France and its food specifically focussing on dishes. Include the paragraph and the list of dishes mentioned the paragraph in seperate fields.
Emphasis:    ['Return the response for the following as a JSON:']
Alpha:       0.1
Max tokens:  500
Temperature: 0.0


## Generate Baseline (No Spotlight)

In [4]:
sampling_params = SamplingParams(
    temperature=temperature,
    max_tokens=max_tokens
)

print("Generating baseline (no attention steering)...\n")
outputs_baseline = llm.generate(
    prompts=[prompt],
    sampling_params=sampling_params,
    use_hook=False  # Disable hooks
)

baseline_text = outputs_baseline[0].outputs[0].text

print("─" * 70)
print("[BASELINE — Without Spotlight]")
print("─" * 70)
print(baseline_text)
print()

Generating baseline (no attention steering)...



Processed prompts: 100%|██████████| 1/1 [00:30<00:00, 30.21s/it, est. speed input: 1.49 toks/s, output: 16.55 toks/s]

──────────────────────────────────────────────────────────────────────
[BASELINE — Without Spotlight]
──────────────────────────────────────────────────────────────────────
 France is a country that is known for its rich culture, history, and cuisine. The French are known for their love of food and their ability to create delicious dishes that are both elegant and satisfying. One of the most famous dishes in France is the classic dish of escargot, which is a dish of snails cooked in garlic butter. Another popular dish in France is the classic dish of ratatouille, which is a dish of vegetables cooked in tomato sauce. Another popular dish in France is the classic dish of coq au vin, which is a dish of chicken cooked in red wine and vegetables. Another popular dish in France is the classic dish of bouillabaisse, which is a dish of fish and seafood cooked in a broth. Another popular dish in France is the classic dish of ratatouille, which is a dish of vegetables cooked in tomato sauce. Ano

## Generate With Spotlight

In [5]:
print(f"Generating with Spotlight (alpha={alpha})...\n")
outputs_spotlight = generate_with_spotlight(
    llm,
    prompts=[prompt],
    emph_strings=emph_strings,
    alpha=alpha,
    temperature=temperature,
    max_tokens=max_tokens,
)

spotlight_text = outputs_spotlight[0].outputs[0].text

print("-" * 70)
print(f"[WITH SPOTLIGHT -- alpha={alpha}]")
print("-" * 70)
print(spotlight_text)
print()

Generating with Spotlight (alpha=0.1)...

Logged run ID.
Created hook flag.


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=1973) [HOOK] Prefill: alpha=0.1, Q=torch.Size([64, 1536]), query_lens=tensor([64], device='cuda:0', dtype=torch.int32)
(EngineCore pid=1973) [BIAS] current=0.1924, target=0.1000, steer=False
(EngineCore pid=1973) [HOOK] output diff: mean=0.000006, max=0.001953
(EngineCore pid=1973) [HOOK] Prefill: alpha=0.1, Q=torch.Size([64, 1536]), query_lens=tensor([64], device='cuda:0', dtype=torch.int32)
(EngineCore pid=1973) [BIAS] current=0.2202, target=0.1000, steer=False
(EngineCore pid=1973) [HOOK] output diff: mean=0.000007, max=0.000488
(EngineCore pid=1973) [HOOK] Prefill: alpha=0.1, Q=torch.Size([64, 1536]), query_lens=tensor([64], device='cuda:0', dtype=torch.int32)
(EngineCore pid=1973) [BIAS] current=0.0892, target=0.1000, steer=True


(EngineCore pid=1973) /home/danish/projects/vLLM-Hook/vllm_hook_plugins/vllm_hook_plugins/utils/spotlight/utils.py:226: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
(EngineCore pid=1973)   torch.tensor(


(EngineCore pid=1973) [BIAS] after steering: 0.0955
(EngineCore pid=1973) [HOOK] output diff: mean=0.001299, max=0.045410
(EngineCore pid=1973) [HOOK] Prefill: alpha=0.1, Q=torch.Size([64, 1536]), query_lens=tensor([64], device='cuda:0', dtype=torch.int32)
(EngineCore pid=1973) [BIAS] current=0.1167, target=0.1000, steer=False
(EngineCore pid=1973) [HOOK] output diff: mean=0.000010, max=0.000977
(EngineCore pid=1973) [HOOK] Prefill: alpha=0.1, Q=torch.Size([64, 1536]), query_lens=tensor([64], device='cuda:0', dtype=torch.int32)
(EngineCore pid=1973) [BIAS] current=0.1233, target=0.1000, steer=False
(EngineCore pid=1973) [HOOK] output diff: mean=0.000011, max=0.000977
(EngineCore pid=1973) [HOOK] Prefill: alpha=0.1, Q=torch.Size([64, 1536]), query_lens=tensor([64], device='cuda:0', dtype=torch.int32)
(EngineCore pid=1973) [BIAS] current=0.0938, target=0.1000, steer=True
(EngineCore pid=1973) [BIAS] after steering: 0.0979
(EngineCore pid=1973) [HOOK] output diff: mean=0.001134, max=0.067

Processed prompts: 100%|██████████| 1/1 [00:28<00:00, 28.46s/it, est. speed input: 2.25 toks/s, output: 10.30 toks/s]

Hooks deactivated.
----------------------------------------------------------------------
[WITH SPOTLIGHT -- alpha=0.1]
----------------------------------------------------------------------
{
  "paragraph": "France is a country known for its rich culinary traditions and diverse cuisine. From the classic French dishes like coq au vin and escargots to more modern creations like croissants and macarons, France has something for everyone. The country's cuisine is characterized by its use of fresh, high-quality ingredients and its emphasis on simplicity and balance. Some of the most famous French dishes include coq au vin, a dish made with chicken, wine, and mushrooms, and escargots, a dish made with snails cooked in garlic butter. Other popular French dishes include ratatouille, a vegetable stew made with tomatoes, zucchini, eggplant, and bell peppers, and bouillabaisse, a fish stew made with a variety of seafood and vegetables. In addition to these classic dishes, France also has a rich 

## Comparison

In [6]:
print("="*70)
print("COMPARISON")
print("="*70)
print(f"\nPrompt: {prompt}")
print(f"Emphasized span(s): {emph_strings}")
print(f"Alpha (target attention): {alpha}")
print()

print("\n" + "─"*70)
print("BASELINE (No Steering)")
print("─"*70)
print(baseline_text)

print("\n" + "─"*70)
print(f"WITH SPOTLIGHT (alpha={alpha})")
print("─"*70)
print(spotlight_text)

print("\n" + "="*70)

COMPARISON

Prompt: Return the response for the following as a JSON: Write a 400 word paragraph on France and its food specifically focussing on dishes. Include the paragraph and the list of dishes mentioned the paragraph in seperate fields.
Emphasized span(s): ['Return the response for the following as a JSON:']
Alpha (target attention): 0.1


──────────────────────────────────────────────────────────────────────
BASELINE (No Steering)
──────────────────────────────────────────────────────────────────────
 France is a country that is known for its rich culture, history, and cuisine. The French are known for their love of food and their ability to create delicious dishes that are both elegant and satisfying. One of the most famous dishes in France is the classic dish of escargot, which is a dish of snails cooked in garlic butter. Another popular dish in France is the classic dish of ratatouille, which is a dish of vegetables cooked in tomato sauce. Another popular dish in France is the